In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/imoex-forecaster'

for sub in [
    'data/processed/train.parquet',
    'data/processed/val.parquet',
    'data/processed/test.parquet',
    'models/word2vec.kv',
]:
    path = os.path.join(DRIVE_DIR, sub)
    print(f"{sub:45s} {'OK' if os.path.exists(path) else 'MISSING'}")

## Чтение данных

In [ ]:
%cd /content
!rm -rf imoex && git clone --quiet https://github.com/mkgrini/diploma-imoex-forecaster.git imoex
%cd /content/imoex
!git log --oneline -3

In [ ]:
!pip install --quiet 'gensim>=4.4' 'pyarrow' 'scikit-learn'

In [ ]:
!mkdir -p data/processed models
!cp $DRIVE_DIR/data/processed/train.parquet data/processed/
!cp $DRIVE_DIR/data/processed/val.parquet   data/processed/
!cp $DRIVE_DIR/data/processed/test.parquet  data/processed/
!cp $DRIVE_DIR/models/word2vec.kv           models/
!ls -la data/processed models | tail -10

## Модель

In [ ]:
import sys, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from gensim.models import KeyedVectors

sys.path.insert(0, '/content/imoex')
from src.ml.dataset import EMBED_DIM, NUMERIC_DIM, NewsLSTMDataset, collate
from src.ml.lstm import NewsLSTM
from src.ml.eval import evaluate, format_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__, '| device:', device)
if device.type == 'cuda':
    print('GPU:   ', torch.cuda.get_device_name(0))
else:
    print('WARN: GPU не найдена — обучение будет на CPU и медленнее')

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64
LR = 0.001
HIDDEN_SIZE = 256
DROPOUT = 0.2
PATIENCE = 5
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

## Загрузка данных и Word2Vec

In [ ]:
kv = KeyedVectors.load('models/word2vec.kv')
assert kv.vector_size == EMBED_DIM
print(f'W2V vocab: {len(kv):,}, dim: {kv.vector_size}')

train_ds = NewsLSTMDataset(Path('data/processed/train.parquet'), kv)
val_ds   = NewsLSTMDataset(Path('data/processed/val.parquet'), kv, fit_state=train_ds.fit_state)
test_ds  = NewsLSTMDataset(Path('data/processed/test.parquet'), kv, fit_state=train_ds.fit_state)
print(f'Сплиты: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

## Инициализация модели

In [ ]:
model = NewsLSTM(
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=1,
    num_numeric=NUMERIC_DIM,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = torch.nn.MSELoss()

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nПараметров: {n_params:,}')

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    preds, trues = [], []
    for text_emb, lengths, numeric, target in loader:
        text_emb = text_emb.to(device)
        numeric  = numeric.to(device)
        y = model(text_emb, lengths, numeric).cpu().numpy()
        preds.append(y)
        trues.append(target.numpy())
    return np.concatenate(preds), np.concatenate(trues)

## Training

Лучший по `val_mse` чекпойнт сохраняется в `models/lstm_best.pt`. История лоссов и метрик копится в `history` для графиков.

In [ ]:
history = {'epoch': [], 'train_mse': [], 'val_mse': [], 'val_dir_acc': [], 'vs_naive': []}
best_val_mse = float('inf')
patience_left = PATIENCE
best_path = Path('models/lstm_best.pt')
scaler_path = Path('models/lstm_scaler.pkl')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    losses = []
    for text_emb, lengths, numeric, target in train_loader:
        text_emb = text_emb.to(device)
        numeric  = numeric.to(device)
        target   = target.to(device)
        optimizer.zero_grad()
        pred = model(text_emb, lengths, numeric)
        loss = loss_fn(pred, target)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    train_mse = float(np.mean(losses))

    val_pred, val_true = predict(model, val_loader, device)
    m = evaluate(val_true, val_pred)
    dt = time.time() - t0

    history['epoch'].append(epoch)
    history['train_mse'].append(train_mse)
    history['val_mse'].append(m['mse'])
    history['val_dir_acc'].append(m['dir_acc'])
    history['vs_naive'].append(m['mse'] / m['mse_naive'])

    print(
        f'epoch {epoch:3d}  train_mse={train_mse:.3e}  '
        f"val_mse={m['mse']:.3e}  val_dir={m['dir_acc']:.1%}  "
        f"vs_naive={m['mse'] / m['mse_naive']:.3f}  ({dt:.1f}s)"
    )

    if m['mse'] < best_val_mse - 1e-12:
        best_val_mse = m['mse']
        patience_left = PATIENCE
        torch.save(model.state_dict(), best_path)
        with scaler_path.open('wb') as f:
            pickle.dump(train_ds.fit_state, f)
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f'\nEarly stopping на эпохе {epoch} (patience exhausted)')
            break

print(f'\nBest val MSE: {best_val_mse:.3e}  → {best_path}')

## Оценка

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['epoch'], history['train_mse'], label='train', marker='o', ms=3)
axes[0].plot(history['epoch'], history['val_mse'],   label='val',   marker='o', ms=3)
axes[0].set_yscale('log'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('MSE (log)')
axes[0].set_title('Loss curves'); axes[0].legend()

axes[1].plot(history['epoch'], history['vs_naive'], marker='o', ms=3, color='tab:red')
axes[1].axhline(1.0, color='gray', ls='--', label='naive baseline')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val_mse / naive_mse')
axes[1].set_title('vs naive (хотим < 1)'); axes[1].legend()

axes[2].plot(history['epoch'], history['val_dir_acc'], marker='o', ms=3, color='tab:green')
axes[2].axhline(0.5, color='gray', ls='--', label='random')
axes[2].set_xlabel('epoch'); axes[2].set_ylabel('val dir_acc')
axes[2].set_title('Directional accuracy'); axes[2].legend()

plt.tight_layout(); plt.show()

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))

metrics_by_split = {}
for name, loader in [('train', train_loader), ('val', val_loader), ('test', test_loader)]:
    pred, true = predict(model, loader, device)
    m = evaluate(true, pred)
    metrics_by_split[name] = (m, pred, true)
    print(format_metrics(name, m))

In [ ]:
_, test_pred, test_true = metrics_by_split['test']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(test_true, label='true', alpha=0.7, lw=0.9)
axes[0].plot(test_pred, label='pred', alpha=0.7, lw=0.9)
axes[0].set_title('Test: true vs predicted hourly return')
axes[0].set_xlabel('index'); axes[0].legend()

axes[1].scatter(test_true, test_pred, s=4, alpha=0.4)
lim = max(abs(test_true).max(), abs(test_pred).max()) * 1.05
axes[1].plot([-lim, lim], [-lim, lim], color='gray', ls='--', lw=0.5)
axes[1].axhline(0, color='gray', lw=0.3); axes[1].axvline(0, color='gray', lw=0.3)
axes[1].set_xlabel('true'); axes[1].set_ylabel('pred')
axes[1].set_title('Pred vs true')

plt.tight_layout(); plt.show()

In [ ]:
export_path = Path('models/lstm_model.pt')
model_cpu = model.cpu().eval()

scripted = torch.jit.script(model_cpu)
scripted.save(str(export_path))
print(f'TorchScript: {export_path}')

## Сохранение артефактов

In [ ]:
import shutil
from datetime import datetime

run_dir = os.path.join(DRIVE_DIR, 'runs', datetime.now().strftime('%Y%m%d_%H%M%S'))
os.makedirs(run_dir, exist_ok=True)

for artifact in ['lstm_best.pt', 'lstm_scaler.pkl', 'lstm_model.pt']:
    src = os.path.join('models', artifact)
    if os.path.exists(src):
        shutil.copy(src, run_dir)
        print(f'  copied {artifact} → {run_dir}')
    else:
        print(f'  {artifact}: not present')

# История обучения тоже сохраним
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(run_dir, 'history.csv'), index=False)
print(f"\nhistory.csv → {run_dir}")
print(f'\nDone. Run dir: {run_dir}')